In [0]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("MyHealthcareApp").getOrCreate()

In [0]:
encounters_file_path = r'abfss://healthcareproject@storageaccgroupc.dfs.core.windows.net/bronze/dbo.encounters.csv'
encounters_df = spark.read.csv(encounters_file_path, header=True, inferSchema=True)
encounters_df.count()

In [0]:
#Deduplication of encounter_id
encounters_Deduplication_df = encounters_df.dropDuplicates(["encounter_id"])
encounters_Deduplication_df.count()


In [0]:
#Admission Datetime Standardization
from pyspark.sql import functions as F

encounters_addmission_datetime_std_df = encounters_Deduplication_df.withColumn(
    "admission_ts",
    F.coalesce(
        F.try_to_timestamp(
            F.col("admission_datetime"),
            F.lit("dd-MM-yyyy HH:mm")
        ),
        F.try_to_timestamp(
            F.col("admission_datetime"),
            F.lit("yyyy-MM-dd'T'HH:mm:ss")
        )
    )
)
encounters_addmission_datetime_std_df.select("admission_datetime", "admission_ts").show(1,truncate=False)
#encounters_addmission_datetime_std_df.groupBy("admission_ts").count().show()

In [0]:
#Discharge Datetime Standardization
from pyspark.sql import functions as F

encounters_discharge_datetime_std_df = encounters_addmission_datetime_std_df.withColumn(
    "discharge_ts",
    F.coalesce(
        F.try_to_timestamp(
            F.col("discharge_datetime"),
            F.lit("dd-MM-yyyy HH:mm")
        ),
        F.try_to_timestamp(
            F.col("discharge_datetime"),
            F.lit("yyyy-MM-dd'T'HH:mm:ss")
        )
    )
)
#encounters_discharge_datetime_std_df.select("discharge_datetime", "discharge_ts").show(10,truncate=False)
#encounters_discharge_datetime_std_df.filter(F.col("discharge_ts").isNull()).show(10,truncate=False)
# encounters_discharge_datetime_std_df.filter(F.col("discharge_ts").isNull()).count()
# encounters_discharge_datetime_std_df.filter(F.col("discharge_ts").isNotNull()).count()

In [0]:
#Derived – LOS(Length Of Stay) Hours
from pyspark.sql import functions as F

encounters_los_hours_final_df = encounters_discharge_datetime_std_df.withColumn(
    "los_hours_final",
    F.when(
        F.col("length_of_stay_hours").isNotNull(),
        F.col("length_of_stay_hours")
    ).when(
        F.col("admission_ts").isNotNull() &
        F.col("discharge_ts").isNotNull(),
        F.round(
            (
                F.unix_timestamp("discharge_ts") -
                F.unix_timestamp("admission_ts")
            ) / 3600,
            1
        )
    ).otherwise(None)
)

# encounters_los_hours_final_df.select(
#     "length_of_stay_hours",
#     "admission_ts",
#     "discharge_ts",
#     "los_hours_final"
# ).show(10,truncate=False)

# encounters_los_hours_final_df.filter(
#     F.col("los_hours_final").isNull()
# ).show(10,truncate=False)

# encounters_los_hours_final_df.filter(
#     F.col("los_hours_final") > 1000
# ).show(truncate=False)

# encounters_los_hours_final_df.filter(
#     F.col("los_hours_final") < 0
# ).show(truncate=False)

encounters_los_hours_final_df.filter(
    F.col("los_hours_final").isNotNull()
).count()

In [0]:
#Derived – LOS Days
from pyspark.sql import functions as F

encounters_los_days_df = encounters_los_hours_final_df.withColumn(
    "los_days",
    F.when(
        F.col("los_hours_final").isNotNull(),
        F.round(
            F.col("los_hours_final") / 24,
            1
        )
    ).otherwise(None)
)

# encounters_los_days_df.filter(
#     F.col("los_hours_final").isNotNull()
# ).select(
#     "los_hours_final",
#     "los_days"
# ).show(5, truncate=False)

# # 5 records where los_hours_final is NULL
# encounters_los_days_df.filter(
#     F.col("los_hours_final").isNull()
# ).select(
#     "los_hours_final",
#     "los_days"
# ).show(5, truncate=False)

In [0]:
#Derived – Year
from pyspark.sql import functions as F

encounters_encounter_year_df = encounters_los_days_df.withColumn(
    "encounter_year",
    F.year("admission_ts")
)

# encounters_encounter_year_df.select(
#     "admission_ts",
#     "encounter_year"
# ).show(10, truncate=False)

# encounters_encounter_year_df.filter(
#     F.col("admission_ts").isNull()
# ).select(
#     "admission_ts",
#     "encounter_year"
# ).show(5, truncate=False)

In [0]:
#Derived – Month
from pyspark.sql import functions as F

encounters_encounter_month_df = encounters_encounter_year_df.withColumn(
    "encounter_month",
    F.month("admission_ts")
)

# encounters_encounter_month_df.select(
#     "admission_ts",
#     "encounter_month"
# ).show(10, truncate=False)

# encounters_encounter_month_df.filter(
#     F.col("admission_ts").isNull()
# ).select(
#     "admission_ts",
#     "encounter_year"
# ).show(5, truncate=False)


In [0]:
#Lookup Standardization
from pyspark.sql import functions as F

encounters_encounter_type_std_df = encounters_encounter_month_df.withColumn(
    "encounter_type_std",
    F.when(
        F.col("encounter_type").isNull(),
        "Unknown"
    ).when(
        F.upper(F.col("encounter_type")) == "OP",
        "Outpatient"
    ).when(
        F.upper(F.col("encounter_type")) == "IP",
        "Inpatient"
    ).when(
        F.upper(F.col("encounter_type")) == "EM",
        "Emergency"
    ).when(
        F.upper(F.col("encounter_type")) == "DAY CARE",
        "Day Care"
    ).otherwise(
        F.initcap(F.col("encounter_type"))
    )
)

# encounters_encounter_type_std_df.select(
#     "encounter_type",
#     "encounter_type_std"
# ).show(20, truncate=False)

In [0]:
#Standardization
from pyspark.sql import functions as F

encounters_encounter_status_std_df = encounters_encounter_type_std_df.withColumn(
    "encounter_status_std",
    F.when(
        F.col("encounter_status").isNull(),
        "Unknown"
    ).otherwise(
        F.initcap(
            F.trim(
                F.col("encounter_status")
            )
        )
    )
)

# encounters_encounter_status_std_df.select(
#     "encounter_status",
#     "encounter_status_std"
# ).show(20, truncate=False)

# encounters_encounter_status_std_df.filter(
#     F.col("encounter_status").isNull()
# ).select(
#     "encounter_status",
#     "encounter_status_std"
# ).show(5, truncate=False)

In [0]:
#Numeric Mapping
from pyspark.sql import functions as F

encounters_triage_level_num_df = encounters_encounter_status_std_df.withColumn(
    "triage_level_num",
    F.when(
        F.col("triage_level").isNull(),
        None
    ).when(
        F.upper(F.col("triage_level")).isin("1", "I", "RED"),
        1
    ).when(
        F.upper(F.col("triage_level")).isin("2", "II", "ORANGE"),
        2
    ).when(
        F.upper(F.col("triage_level")).isin("3", "III", "YELLOW"),
        3
    ).when(
        F.upper(F.col("triage_level")).isin("4", "IV", "GREEN"),
        4
    ).when(
        F.upper(F.col("triage_level")).isin("5", "V", "BLUE"),
        5
    ).otherwise(None)
)

# encounters_triage_level_num_df.select(
#     "triage_level",
#     "triage_level_num"
# ).show(20, truncate=False)

In [0]:
#Boolean Normalisation
from pyspark.sql import functions as F

encounters_readmission_flag_std_df = encounters_triage_level_num_df.withColumn(
    "readmission_flag_std",
    F.when(
        F.col("readmission_flag").isNull(),
        0
    ).when(
        F.upper(F.col("readmission_flag")).isin("YES", "Y", "1"),
        1
    ).when(
        F.upper(F.col("readmission_flag")).isin("NO", "N", "0"),
        0
    ).otherwise(None)
)

# encounters_readmission_flag_std_df.select(
#     "readmission_flag",
#     "readmission_flag_std"
# ).show(20, truncate=False)

In [0]:
#Text Cleansing
from pyspark.sql import functions as F

encounters_chief_complaint_clean_df = encounters_readmission_flag_std_df.withColumn(
    "chief_complaint_clean",
    F.trim(
        F.regexp_replace(
            F.col("chief_complaint"),
            r"[^a-zA-Z0-9 ,.]",
            ""
        )
    )
)

# encounters_chief_complaint_clean_df.select(
#     "chief_complaint",
#     "chief_complaint_clean"
# ).show(20, truncate=False)

In [0]:
#Casing Standardization
from pyspark.sql import functions as F

encounters_department_std_df = encounters_chief_complaint_clean_df.withColumn(
    "department_std",
    F.when(
        F.col("department").isNull(),
        "Unknown"
    ).otherwise(
        F.initcap(
            F.trim(
                F.col("department")
            )
        )
    )
)
# encounters_department_std_df.select(
#     "department",
#     "department_std"
# ).show(20, truncate=False)

In [0]:
#Casing Standardization
from pyspark.sql import functions as F

encounters_ward_std_df = encounters_department_std_df.withColumn(
    "ward_std",
    F.upper(
        F.trim(
            F.col("ward")
        )
    )
)
# encounters_ward_std_df.select(
#     "ward",
#     "ward_std"
# ).show(20, truncate=False)

In [0]:
#Range Validation
from pyspark.sql import functions as F

encounters_total_charges_valid_df = encounters_ward_std_df.withColumn(
    "total_charges_valid",
    F.when(
        F.col("total_charges") > 0,
        F.col("total_charges")
    ).otherwise(None)
).withColumn(
    "dq_chrg",
    F.when(
        F.col("total_charges") > 0,
        0
    ).otherwise(1)
)

# encounters_total_charges_valid_df.select(
#     "total_charges",
#     "total_charges_valid",
#     "dq_chrg"
# ).show(20, truncate=False)

In [0]:
#Standardization
from pyspark.sql import functions as F

encounters_discharge_disp_std_df = encounters_total_charges_valid_df.withColumn(
    "discharge_disp_std",
    F.when(
        F.col("discharge_disposition").isNull(),
        "Unknown"
    ).otherwise(
        F.initcap(
            F.trim(
                F.col("discharge_disposition")
            )
        )
    )
)

# encounters_discharge_disp_std_df.select(
#     "discharge_disposition",
#     "discharge_disp_std"
# ).show(20, truncate=False)

In [0]:
#Casing Standardization

from pyspark.sql import functions as F

encounters_source_system_std_df = encounters_discharge_disp_std_df.withColumn(
    "source_system_std",
    F.when(
        F.col("source_system").isNull(),
        "UNKNOWN"
    ).otherwise(
        F.upper(
            F.trim(
                F.col("source_system")
            )
        )
    )
)

# encounters_source_system_std_df.select(
#     "source_system",
#     "source_system_std"
# ).show(20, truncate=False)

In [0]:
#Standardization
from pyspark.sql import functions as F

encounters_encounter_subtype_std_df = encounters_source_system_std_df.withColumn(
    "encounter_subtype_std",
    F.initcap(
        F.trim(
            F.col("encounter_subtype")
        )
    )
)
# encounters_encounter_subtype_std_df.select(
#     "encounter_subtype",
#     "encounter_subtype_std"
# ).show(20, truncate=False)

In [0]:
#Data Quality Flag
from pyspark.sql import functions as F

encounters_dq_encounter_flag_df = encounters_encounter_subtype_std_df.withColumn(
    "dq_encounter_flag",
    F.when(
        (F.col("admission_ts").isNull()) |
        (F.col("encounter_type_std") == "Unknown") |
        (F.col("total_charges_valid").isNull()),
        1
    ).otherwise(0)
)

# encounters_dq_encounter_flag_df.select(
#     "admission_ts",
#     "encounter_type_std",
#     "total_charges_valid",
#     "dq_encounter_flag"
# ).show(20, truncate=False)

In [0]:
# ----------------------------------------
#  Write to Silver layer in delta format
# ----------------------------------------


TABLE_NAME = "encounters"  

silver_table_path = f"abfss://healthcareproject@storageaccgroupc.dfs.core.windows.net/silver/{TABLE_NAME}"

# ── OVERWRITE (Full Load) ────────────────────────────────────
encounters_dq_encounter_flag_df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .save(silver_table_path)

print(f"✅ Data written to Silver layer: {silver_table_path}")